In [1]:
import os
import pickle as pkl
import sys
from pathlib import Path

from pydeseq2.dds import DeseqDataSet
from pydeseq2.default_inference import DefaultInference
from pydeseq2.ds import DeseqStats
from pydeseq2.utils import load_example_data

import pandas as pd

# set project root directory
_root = Path().resolve()
if (_root / "src").is_dir():
    pass  # cwd is project root
else:
    _root = _root.parent  # cwd is notebooks/
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

# set output directory for results
SAVE = False
if SAVE:    
    _output_dir = _root / "results"
    _output_dir.mkdir(exist_ok=True)

# Load in Data
Downloaded from https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE47363

In [10]:
# load in GSE47363 dataset from txt file
def load_data(filename):
    '''
    Loads in data from a txt file into a Pandas DataFrame.

    Parameters
    ----------
    filename : str
        File path for the dataset.
    
    Returns
    -------
    data : Pandas DataFrame
        DataFrame containing the dataset.
    '''
    filename_str = str(filename)

    if not os.path.isfile(filename):
        raise FileNotFoundError(f"Data file not found: {filename}")
    
    if "metadata" in filename_str:
        data = pd.read_csv(filename, sep="\t", skiprows=8)
    elif "non-normalized" in filename_str:
        data = pd.read_csv(filename, sep="\t", skiprows=4, index_col="ID_REF")
    else:
        raise ValueError(f"Unrecognized filename: {filename}")
    return data

# load in counts
filename = _root / "data" / "GSE47363_non-normalized.txt"
df = load_data(filename)
print(f"Data shape: {df.shape}")

# load in metadata
filename = _root / "data" / "GSE47363_metadata.txt"
metadata = load_data(filename)
print(f"Metadata shape: {metadata.shape}")

Data shape: (47323, 12)
Metadata shape: (48240, 28)


# Pre-Processing for pydeseq2

In [16]:
def pre_processing(data):
    '''
    Drops p-value columns, transposes data for pydeseq2, and rounds data to the nearest integer.

    Parameters
    ----------
    data : Pandas DataFrame
        Cleaned and transposed DataFrame (samples as rows, genes as columns).
    
    '''
    # drop p-value columns and transpose data for pydeseq2
    cols_to_drop = [col for col in data.columns if "PVAL" in col.upper()]
    data = data.drop(columns=cols_to_drop)
    data = data.transpose()

    # pydeseq2 expects integer counts, so round the data to the nearest integer
    data = data.round().astype(int)
    return data

df = pre_processing(df)
print(f"Data shape: {df.shape}")
df.head()

Data shape: (6, 47323)


ID_REF,ILMN_1343291,ILMN_1343295,ILMN_1651199,ILMN_1651209,ILMN_1651210,ILMN_1651221,ILMN_1651228,ILMN_1651229,ILMN_1651230,ILMN_1651232,...,ILMN_3311145,ILMN_3311150,ILMN_3311155,ILMN_3311160,ILMN_3311165,ILMN_3311170,ILMN_3311175,ILMN_3311180,ILMN_3311185,ILMN_3311190
miR-neg_rep1,34463,18570,113,123,133,134,2132,321,129,149,...,136,148,135,140,173,120,125,126,155,175
miR-542-3p_rep1,30407,21196,117,144,150,159,564,349,130,148,...,126,164,130,141,187,144,135,164,137,147
miR-neg_rep2,33630,17133,137,131,113,142,1701,304,126,140,...,122,159,131,125,153,133,107,153,133,134
miR-542-3p_rep2,29308,18254,124,121,122,122,593,319,117,125,...,113,117,129,107,151,109,111,138,122,126
miR-neg_rep3,36107,20843,104,152,127,147,2215,341,113,173,...,123,125,125,136,174,129,140,137,144,150


# Data Filtering

In [21]:
def filter_data(data, metadata_df):
    '''
    Filters out genes with low read counts and samples with missing condition labels.
    Assumes that data was already transposed.

    Parameters
    ----------
    df : Pandas DataFrame
        DataFrame containing the gene expression counts.
    
    metadata : Pandas DataFrame
        DataFrame containing the sample metadata.
    
    Returns
    -------
    df : Pandas DataFrame
        Filtered DataFrame containing the gene expression counts.
    
    metadata : Pandas DataFrame
        Filtered DataFrame containing the sample metadata.
    '''

    # make sure metadata and data rows match up (i.e. samples in the same order)
    metadata_df = metadata_df.reindex(data.index)

    # create condition column in metadata based on characteristics of the samples (e.g. "control" vs "disease")
    metadata_df['condition'] = [
        'treatment' if '542-3p' in str(idx) else 'control' 
        for idx in metadata_df.index
    ]

    # filter out NaN samples (samples with missing condition labels) from both the data and metadata
    samples_to_keep = ~metadata_df.condition.isna()
    data = data.loc[samples_to_keep]
    metadata_df = metadata_df.loc[samples_to_keep]

    # filter out genes that have less than 10 read counts in total (eliminate noise/genes with very low expression across all samples)
    genes_to_keep = data.columns[data.sum(axis=0) >= 10]
    data = data[genes_to_keep]

    return data, metadata_df

df, metadata = filter_data(df, metadata)
print(f"Filtered data shape: {df.shape}")
print(f"Filtered metadata shape: {metadata.shape}")
print(metadata['condition'].value_counts()) # expecting 3 each of "control" and "treatment" samples

Filtered data shape: (6, 47323)
Filtered metadata shape: (6, 29)
condition
control      3
treatment    3
Name: count, dtype: int64


# Single Factor Analysis

In [27]:
def compare_groups(dds):
    '''
    Compares gene expression between control and treatment groups using pydeseq2.

    Parameters
    ----------
    dds : DeseqDataSet
        DeseqDataSet object containing the fitted model parameters.
    
    Returns
    -------
    results : Pandas DataFrame
        DataFrame containing the results of the differential expression analysis.
    '''
    inference = DefaultInference(n_cpus=8)
    dds = DeseqDataSet( # fits dispersion and LFC parameters from data and stores them
        counts=df,
        metadata=metadata,
        design="~condition",
        refit_cooks=True,
        inference=inference,
        n_cpus=None) # use all available CPUs

    dds.deseq2()

    if SAVE:
        with open(os.path.join(_output_dir, "dds.pkl"), "wb") as f:
            pkl.dump(dds, f)

    print(dds)

    # compute results for the contrast of interest (treatment vs control)
    results = DeseqStats(dds, contrast=["condition", "control", "treatment"], inference=inference)
    results.summary() # fit p-values and log fold changes
    return results

results = compare_groups(df)
results_df = results.results_df
results_df.head()

Fitting size factors...
... done in 0.01 seconds.



Using None as control genes, passed at DeseqDataSet initialization


Fitting dispersions...
... done in 3.54 seconds.

Fitting dispersion trend curve...
/opt/conda/envs/anna_analysis/lib/python3.10/site-packages/pydeseq2/dds.py:807: UserWarning: The dispersion trend curve fitting did not converge. Switching to a mean-based dispersion trend.
  self._fit_parametric_dispersion_trend(vst)
... done in 0.49 seconds.

Fitting MAP dispersions...
... done in 7.96 seconds.

Fitting LFCs...
... done in 3.31 seconds.

Calculating cook's distance...
... done in 0.01 seconds.

Replacing 0 outlier genes.

Running Wald tests...


AnnData object with n_obs × n_vars = 6 × 47323
    obs: 'Species', 'Source', 'Search_Key', 'Transcript', 'ILMN_Gene', 'Source_Reference_ID', 'RefSeq_ID', 'Unigene_ID', 'Entrez_Gene_ID', 'GI', 'Accession', 'Symbol', 'Protein_Product', 'Probe_Id', 'Array_Address_Id', 'Probe_Type', 'Probe_Start', 'Probe_Sequence', 'Chromosome', 'Probe_Chr_Orientation', 'Probe_Coordinates', 'Cytoband', 'Definition', 'Ontology_Component', 'Ontology_Process', 'Ontology_Function', 'Synonyms', 'Obsolete_Probe_Id', 'condition', 'size_factors', 'replaceable'
    var: '_normed_means', 'non_zero', '_MoM_dispersions', 'genewise_dispersions', '_genewise_converged', 'fitted_dispersions', 'MAP_dispersions', '_MAP_converged', 'dispersions', '_outlier_genes', '_LFC_converged', 'replaced', 'refitted', '_pvalue_cooks_outlier'
    uns: 'mean_disp', 'disp_function_type', '_squared_logres', 'prior_disp_var'
    obsm: 'design_matrix', '_mu_LFC', '_hat_diagonals'
    varm: 'LFC'
    layers: 'normed_counts', '_mu_hat', 'cooks'


... done in 1.74 seconds.



,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
ID_REF,,,,,,
ILMN_1343291,32246.259552,0.198213,0.093667,2.116151,0.034332,0.103079
ILMN_1343295,19091.918735,-0.063025,0.100745,-0.625587,0.531586,0.714518
ILMN_1651199,119.120052,-0.039371,0.158845,-0.247859,0.804244,NaN
ILMN_1651209,132.716812,0.031198,0.147336,0.211750,0.832302,NaN
ILMN_1651210,131.751769,-0.186587,0.146782,-1.271180,0.203664,NaN


In [ ]:
# Next steps:
# - mapping gene IDs to gene names
# - thresholding results based on adjusted p-value and log fold change to get list of differentially expressed genes (e.g. adj p-value < 0.05 and abs(log fold change) > 1)
# - visualizations (e.g. volcano plot, heatmap of top differentially expressed genes)
# - output three col file for pathway enrichment analysis (gene symbol, logFC, adj_p-value) - should be .tsv